# IS 445 Homework: UFO Sightings with Altair

Two plots of the UFO sightings dataset, exported as vegalite JSON for my Jekyll page.

1. How the shapes people report have changed over time (hover to highlight a shape).
2. When sightings happen by month and hour of day (dropdown to filter by shape).

In [1]:
import pandas as pd
import altair as alt
import os

## Read in the data

The file has no header row, so I'm passing in column names (same ones we used in class) and reading the `date` column in as timestamps.

In [ ]:
ufo_url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv'

ufos = pd.read_csv(ufo_url, names=['date', 'city', 'state', 'country', 'shape', 'duration_seconds', 'duration', 'comment', 'report_date','latitude', 'longitude'], parse_dates=['date'])
ufos.head()

,date,city,state,country,shape,duration_seconds,duration,comment,report_date,latitude,longitude
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,1949-10-10 21:00:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,1955-10-10 17:00:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,1956-10-10 21:00:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [3]:
len(ufos)

80332

## Clean up

First, pull the year, month, and hour out of the timestamps
Then, drop rows with no shape listed since both plots are split by shape
Finally, keep 1950.

In [ ]:
ufos['year']= ufos['date'].dt.year
ufos['month']= ufos['date'].dt.month
ufos['hour']= ufos['date'].dt.hour

ufos_clean =ufos.dropna(subset=['shape'])
ufos_clean = ufos_clean.loc[ufos_clean['year']>= 1950].copy()
len(ufos_clean)

78283

There are 29 different shapes and most of them are tiny. Im keeping six shapes that show a clear trend over time and putting everything else (including the dataset's own "other" and "unknown" labels) into one "all other shapes" group.

In [5]:
ufos_clean['shape'].value_counts()

shape
light        16556
triangle      7863
circle        7599
fireball      6205
other         5642
unknown       5576
sphere        5379
disk          5178
oval          3724
formation     2452
cigar         2046
changing      1962
flash         1327
rectangle     1293
cylinder      1278
diamond       1178
chevron        951
egg            759
teardrop       750
cone           316
cross          233
delta            7
round            2
crescent         2
pyramid          1
flare            1
hexagon          1
dome             1
changed          1
Name: count, dtype: int64

In [ ]:
keep_shapes = ['light','fireball', 'circle', 'triangle','disk', 'cigar']

ufos_clean['shape_group'] = 'all other shapes'
for s in keep_shapes:
    ufos_clean.loc[ufos_clean['shape'] == s, 'shape_group'] =s
ufos_clean['shape_group'].value_counts()

shape_group
all other shapes    32836
light               16556
triangle             7863
circle               7599
fireball             6205
disk                 5178
cigar                2046
Name: count, dtype: int64

## Plot 1: Share of reports by shape, in 5 year periods

Single years before the mid 90s only have a couple hundred reports so the yearly shares jump around a lot. Grouping into 5 year periods smooths that out. For each period I count reports per shape group and divide by the period total to get a percent.

In [ ]:
ufos_clean['period'] = (ufos_clean['year']// 5) * 5
ufos_clean['reports'] = 1
shape_counts = ufos_clean.groupby(['period', 'shape_group'])['reports'].sum().unstack()
shape_counts

shape_group,all other shapes,cigar,circle,disk,fireball,light,triangle
period,,,,,,,
1950,56,12,23,64,7,11,4
1955,81,21,24,61,13,34,4
1960,141,25,37,97,6,33,14
1965,357,70,96,221,29,117,44
1970,336,49,101,233,22,116,56
1975,547,75,133,313,37,176,140
1980,408,42,69,157,25,109,106
1985,529,46,85,134,32,118,159
1990,620,46,103,166,51,208,252


In [ ]:
period_totals = shape_counts.sum(axis=1)
shape_pct = shape_counts.copy()
for s in shape_pct.columns:
    shape_pct[s] = (100 * shape_counts[s]/period_totals).round(1)
shape_share = shape_pct.reset_index().melt('period', var_name='shape_group', value_name='percent')
shape_share['years'] = shape_share['period'].astype('str') + '-' + (shape_share['period'] + 4).astype('str')
shape_share['period'] = pd.to_datetime(shape_share['period'].astype('int'), format='%Y')
shape_share.head(10)

,period,shape_group,percent,years
0,1950-01-01,all other shapes,31.6,1950-1954
1,1955-01-01,all other shapes,34.0,1955-1959
2,1960-01-01,all other shapes,39.9,1960-1964
3,1965-01-01,all other shapes,38.2,1965-1969
4,1970-01-01,all other shapes,36.8,1970-1974
5,1975-01-01,all other shapes,38.5,1975-1979
6,1980-01-01,all other shapes,44.5,1980-1984
7,1985-01-01,all other shapes,48.0,1985-1989
8,1990-01-01,all other shapes,42.9,1990-1994
9,1995-01-01,all other shapes,41.7,1995-1999


The last period is 2010 - 2014, but the data stops in May 2014.

In [9]:
ufos_clean['date'].max()

Timestamp('2014-05-08 18:45:00')

In [ ]:
hover = alt.selection_point(fields=['shape_group'], on='mousemove')
shapes_over_time = alt.Chart(shape_share).mark_area().encode(x=alt.X('period:T', axis=alt.Axis(format='%Y',
                                                             title='Start of 5-year period')),
                                                             y=alt.Y('percent:Q', 
                                                             scale=alt.Scale(domain=[0, 100]), 
                                                             axis=alt.Axis(title='Share of reports (%)')), 
                                                             color=alt.Color('shape_group:N', scale=alt.Scale(scheme='tableau10'), title='Reported shape'), opacity=alt.condition(hover, alt.value(1.0), alt.value(0.25)), tooltip=['shape_group', 'years', 'percent']).add_params(
    hover
).properties(
    width=600,
    height=350,
    title='What shape did people say the UFO was? (1950-2014)'
)
shapes_over_time

alt.Chart(...)

## Plot 2: When sightings happen (month x hour of day)

I count reports for every month and hour, split by shape group, then add an "all shapes" column with the total. Melting that back into a long table gives one row per shape group, month, and hour, which is what the dropdown filters on. It's about 2,300 rows instead of ~78,000.

In [ ]:
time_table = ufos_clean.groupby(['month', 'hour', 'shape_group'])['reports'].sum().unstack()
time_table['all shapes'] = time_table.sum(axis=1)
time_counts = time_table.reset_index().melt(['month', 'hour'], var_name='shape_group', value_name='reports')
time_counts.head()

,month,hour,shape_group,reports
0,1,0,all other shapes,214.0
1,1,1,all other shapes,93.0
2,1,2,all other shapes,66.0
3,1,3,all other shapes,59.0
4,1,4,all other shapes,53.0


In [12]:
len(time_counts)

2304

In [ ]:
shape_dropdown = alt.binding_select(options=['all shapes'] + keep_shapes + ['all other shapes'], name='Shape: ')
shape_select = alt.selection_point(fields=['shape_group'], bind=shape_dropdown, value='all shapes')
time_heatmap = alt.Chart(time_counts).mark_rect().encode(
    x=alt.X('hour:O', title='Hour of day (0 = midnight)'),
    y=alt.Y('month:O', title='Month (1 = January)'),
    color=alt.Color('reports:Q', scale=alt.Scale(scheme='yelloworangered'), title='Reports'),
    tooltip=['month', 'hour', 'reports']
).transform_filter(
    shape_select
).add_params(
    shape_select
).properties(
    width=600,
    height=300,
    title='UFO reports by month and hour of day'
)

time_heatmap

alt.Chart(...)

A couple of quick checks on what the heatmap shows. First, the busiest month and hour blocks overall. Second, how many fireball reports land on July 4th, since July looks extra hot when you filter to fireball.

In [14]:
time_counts.loc[time_counts['shape_group'] == 'all shapes'].sort_values('reports', ascending=False).head(10)

,month,hour,shape_group,reports
2182,7,22,all shapes,1990.0
2181,7,21,all shapes,1597.0
2206,8,22,all shapes,1464.0
2205,8,21,all shapes,1462.0
2158,6,22,all shapes,1363.0
2183,7,23,all shapes,1306.0
2229,9,21,all shapes,1221.0
2228,9,20,all shapes,1109.0
2157,6,21,all shapes,1097.0
2159,6,23,all shapes,1058.0


In [15]:
len(ufos_clean.loc[(ufos_clean['shape'] == 'fireball') & (ufos_clean['month'] == 7) & (ufos_clean['date'].dt.day == 4)])

193

## Save the specs for Jekyll

After these run, move the JSON files into `assets/json` in the Jekyll repo.

In [ ]:
myJekyllDir = '../assets/json/'
shapes_over_time.save(myJekyllDir + 'ufo_shapes_over_time.json')
time_heatmap.save(myJekyllDir + 'ufo_time_heatmap.json')